[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C67_LLM_Judge_Course/03_meta_eval/03_meta_eval.ipynb)

# 03 · 元评测与校准（人类上界 / kappa / 样本级 vs 系统级 / ECE / 成本-一致性 / 漂移）

目标：把「这个 judge 能不能用」从一句感觉，变成一份**可以贴进报告的元评测卡**。

本 notebook 你会亲手实现：
1. **人类上界与天花板归一化** —— 把 82% 翻译成「达到人类水平的 94%」
2. **accuracy / kappa / Krippendorff** —— 三个系数在不均衡分布上的分歧，以及 kappa 悖论
3. **样本级 vs 系统级一致性** —— 一致率 70% 的 judge 为什么能给出正确的排名
4. **校准** —— ECE、可靠性图（文本版）、温度缩放，以及「用 swap 一致性当置信度」
5. **成本-一致性前沿** —— 便宜 judge + 大样本 vs 贵 judge + 小样本
6. **CUSUM 漂移检测** —— 抓「每次只掉 0.5 个点、连续掉十次」的缓慢漂移

> 心智模型：**一致率必须配着人类上界一起读；样本级差不代表系统级差；
> 但这条论证的前提是「误差是随机的」——所以偏差探针永远排在一致率分析之前。**

## 1 · 人类上界与天花板归一化

In [ ]:
import math, json
from collections import Counter, defaultdict
import numpy as np

def p_from_hh(a_hh):
    """由人类之间的一致率反解「单个标注者与潜在真值一致的概率」p_H。
    二元任务: a_hh = p^2 + (1-p)^2  →  p = (1 + sqrt(2*a_hh - 1)) / 2"""
    if a_hh < 0.5:
        return float('nan')
    return (1 + math.sqrt(2 * a_hh - 1)) / 2

def p_judge_from_jh(a_jh, p_h):
    """由 judge-human 一致率与 p_H 反解 judge 的真实准确率 p_J。
    a_jh = p_J*p_H + (1-p_J)*(1-p_H)  →  p_J = (a_jh - (1-p_H)) / (2*p_H - 1)"""
    denom = 2 * p_h - 1
    if abs(denom) < 1e-9:
        return float('nan')
    return (a_jh - (1 - p_h)) / denom

def ceiling_normalized(a_jh, a_hh, a_chance=0.5):
    return (a_jh - a_chance) / (a_hh - a_chance)

print(f"{'a_HH':>8}{'p_H':>8}{'a_JH':>8}{'p_J':>8}{'天花板归一化':>14}")
for a_hh, a_jh in [(0.95, 0.90), (0.84, 0.82), (0.75, 0.70), (0.70, 0.68)]:
    ph = p_from_hh(a_hh)
    pj = p_judge_from_jh(a_jh, ph)
    cn = ceiling_normalized(a_jh, a_hh)
    print(f'{a_hh:>8.2f}{ph:>8.3f}{a_jh:>8.2f}{pj:>8.3f}{cn:>14.3f}')

ph = p_from_hh(0.84)
pj = p_judge_from_jh(0.82, ph)
assert abs(p_from_hh(1.0) - 1.0) < 1e-9
assert 0.85 < pj < 1.0, 'a_HH=0.84 时，a_JH=0.82 反解出的 judge 真实准确率很高'
assert abs(ceiling_normalized(0.82, 0.84) - 0.9412) < 1e-3
print(f'\n人类一致率 0.84 → 单个标注者准确率 p_H = {ph:.3f}')
print(f'judge 一致率 0.82 → judge 真实准确率 p_J = {pj:.3f}  ← 比 0.82 高得多')
print('✅ 「82 分」和「达到人类水平的 94%」是同一个数字的两种读法，')
print('   而它们会导向完全不同的资源分配决策。')

In [ ]:
# 用模拟验证反解是对的：设定真实的 p_H 与 p_J，看能不能还原出来
rng = np.random.default_rng(0)
N = 200000
truth = rng.integers(0, 2, N)
P_H, P_J = 0.88, 0.93
h1 = np.where(rng.random(N) < P_H, truth, 1 - truth)
h2 = np.where(rng.random(N) < P_H, truth, 1 - truth)
jg = np.where(rng.random(N) < P_J, truth, 1 - truth)

a_hh_obs = float((h1 == h2).mean())
a_jh_obs = float((jg == h1).mean())
ph_hat = p_from_hh(a_hh_obs)
pj_hat = p_judge_from_jh(a_jh_obs, ph_hat)
print(f'真实 p_H={P_H:.3f} → 估计 {ph_hat:.3f}   (观测 a_HH={a_hh_obs:.4f})')
print(f'真实 p_J={P_J:.3f} → 估计 {pj_hat:.3f}   (观测 a_JH={a_jh_obs:.4f})')
assert abs(ph_hat - P_H) < 0.01 and abs(pj_hat - P_J) < 0.01
print('\n✅ 反解在这个模型下是准确的。')
print('   ⚠️ 但它假设了「标注者的错误互相独立」——如果两个标注员共享同样的误解，')
print('      a_HH 会虚高，反解出的 p_H 也会虚高。这是这个方法唯一的软肋。')

## 2 · accuracy / kappa / Krippendorff：不均衡分布上的分歧

In [ ]:
def cohen_kappa(a, b):
    a, b = np.asarray(a), np.asarray(b)
    cats = sorted(set(a.tolist()) | set(b.tolist()))
    po = float((a == b).mean())
    pe = sum(float((a == c).mean()) * float((b == c).mean()) for c in cats)
    return (po - pe) / (1 - pe) if abs(1 - pe) > 1e-12 else float('nan')

def krippendorff_alpha_nominal(ratings):
    """名义数据的 Krippendorff α。ratings: (n_raters, n_items)，np.nan 表示缺失。
    α = 1 - D_o / D_e，D_o 是观测不一致，D_e 是期望不一致。"""
    R = np.asarray(ratings, dtype=float)
    n_items = R.shape[1]
    values = [v for v in np.unique(R[~np.isnan(R)])]
    # 观测不一致
    num = 0.0
    den = 0.0
    all_vals = []
    for j in range(n_items):
        col = R[:, j]
        col = col[~np.isnan(col)]
        m = len(col)
        if m < 2:
            continue
        all_vals.extend(col.tolist())
        cnt = Counter(col.tolist())
        # 该 item 内不同值的配对数
        pairs_diff = m * (m - 1) - sum(c * (c - 1) for c in cnt.values())
        num += pairs_diff / (m - 1)
        den += m
    if den == 0:
        return float('nan')
    D_o = num / den
    total = Counter(all_vals)
    n_all = sum(total.values())
    same = sum(c * (c - 1) for c in total.values())
    D_e = (n_all * (n_all - 1) - same) / (n_all - 1) / n_all * (n_all / n_all)
    D_e = 1 - same / (n_all * (n_all - 1))
    return 1 - D_o / D_e if D_e > 0 else float('nan')

rng = np.random.default_rng(3)
n = 4000
# 场景 A：类别均衡
t_bal = rng.integers(0, 2, n)
a1 = np.where(rng.random(n) < 0.90, t_bal, 1 - t_bal)
a2 = np.where(rng.random(n) < 0.90, t_bal, 1 - t_bal)
# 场景 B：极度不均衡（92% 都是「通过」）
t_imb = (rng.random(n) < 0.92).astype(int)
b1 = np.where(rng.random(n) < 0.90, t_imb, 1 - t_imb)
b2 = np.where(rng.random(n) < 0.90, t_imb, 1 - t_imb)

for name, x, y in [('类别均衡', a1, a2), ('极度不均衡(92%通过)', b1, b2)]:
    acc = float((x == y).mean())
    kap = cohen_kappa(x, y)
    alpha = krippendorff_alpha_nominal(np.vstack([x, y]).astype(float))
    print(f'{name:<22} 原始一致率 {acc:.3f} | kappa {kap:.3f} | Krippendorff α {alpha:.3f}')

acc_b = float((b1 == b2).mean())
kap_b = cohen_kappa(b1, b2)
assert acc_b > 0.80 and kap_b < 0.55
print('\n✅ kappa 悖论现场：不均衡分布上原始一致率 > 0.80，但 kappa 只有 0.5 上下。')
print('   kappa 没算错——它在说「你们的一致大部分可以由『都爱说通过』解释」。')
print('   正确反应不是换个好看的系数，而是承认这个样本分布几乎没有区分度，去找更均衡的样本。')

## 3 · 样本级 vs 系统级：一致率 70% 的 judge 能给出正确排名吗

In [ ]:
def system_level_experiment(p_agree, delta, n_items=400, n_trials=1000, seed=0):
    """A 的真实胜率是 0.5 + delta，judge 单条一致率 p_agree。
    返回 (judge 选对 A 的比例, 观测到的平均胜率偏离 0.5 的幅度)。"""
    rng = np.random.default_rng(seed)
    correct, gaps = 0, []
    for _ in range(n_trials):
        truth = (rng.random(n_items) < 0.5 + delta).astype(int)       # A 真实胜出的样本
        obs = np.where(rng.random(n_items) < p_agree, truth, 1 - truth)
        gaps.append(obs.mean() - 0.5)
        if obs.mean() > 0.5:
            correct += 1
    return correct / n_trials, float(np.mean(gaps))

print(f"{'单条一致率':>12}{'真实差距':>10}{'系统级选对率':>14}{'观测到的差距':>14}")
for p_ in [0.95, 0.85, 0.70, 0.60]:
    for d_ in [0.10]:
        c_, g_ = system_level_experiment(p_, d_, n_items=400, seed=1)
        print(f'{p_:>12.0%}{d_:>10.0%}{c_:>14.1%}{g_:>14.1%}')

c70, g70 = system_level_experiment(0.70, 0.10, n_items=400, seed=1)
c95, g95 = system_level_experiment(0.95, 0.10, n_items=400, seed=1)
assert c70 > 0.85, '一致率 70% 时系统级仍然大概率选对'
assert g70 < g95, '一致率低会把观测到的差距压缩'
print(f'\n✅ 一致率只有 70% 的 judge，在 400 条样本上仍以 {c70:.0%} 的概率选对模型。')
print(f'   代价是差距被压缩了：真实 10 个点被观测成 {g70:.1%}（压缩系数 2p-1 = {2*0.70-1:.1f}）。')
print('   ⚠️ 这条论证的**唯一前提是误差随机**。系统偏差不会被聚合平均掉——')
print('      所以偏差探针（模块 02）永远排在一致率分析之前。')

In [ ]:
# 系统级所需样本量：可靠性取决于 (2p-1)*delta*sqrt(n)
def n_for_system_level(p_agree, delta, target_z=2.0):
    eff = (2 * p_agree - 1) * delta          # 被压缩后的观测差距
    if eff <= 0:
        return float('inf')
    return math.ceil((target_z ** 2) * 0.25 / (eff ** 2))

print(f"{'一致率':>10}{'真实差距 10%':>16}{'真实差距 5%':>16}{'真实差距 2%':>16}")
for p_ in [0.95, 0.85, 0.70, 0.60]:
    row = ''.join(f'{n_for_system_level(p_, d):>16,}' for d in [0.10, 0.05, 0.02])
    print(f'{p_:>10.0%}{row}')

assert n_for_system_level(0.70, 0.10) > n_for_system_level(0.95, 0.10)
assert n_for_system_level(0.95, 0.02) > n_for_system_level(0.95, 0.10)
print('\n✅ 一致率从 95% 掉到 70%，所需样本量涨约 5 倍——')
print('   但如果便宜的 judge 让你能跑 10 倍的样本，这笔账仍然是划算的（第 5 节展开）。')

## 4 · 校准：ECE、可靠性图与温度缩放

In [ ]:
def expected_calibration_error(conf, correct, n_bins=10):
    conf = np.asarray(conf, dtype=float)
    correct = np.asarray(correct, dtype=float)
    edges = np.linspace(0, 1, n_bins + 1)
    ece, rows = 0.0, []
    for i in range(n_bins):
        m = (conf > edges[i]) & (conf <= edges[i + 1]) if i > 0 else (conf >= edges[i]) & (conf <= edges[i + 1])
        if m.sum() == 0:
            continue
        acc, cf, w = correct[m].mean(), conf[m].mean(), m.mean()
        ece += w * abs(acc - cf)
        rows.append((edges[i], edges[i + 1], int(m.sum()), float(cf), float(acc)))
    return float(ece), rows

def reliability_table(rows):
    print(f"{'置信度区间':>14}{'n':>7}{'平均置信':>10}{'实际准确':>10}{'差':>8}  可靠性图")
    for lo, hi, cnt, cf, acc in rows:
        bar = ' ' * int(cf * 30) + ('▲' if acc > cf else ('▼' if acc < cf else '●'))
        print(f'  [{lo:.1f},{hi:.1f}]{cnt:>7}{cf:>10.3f}{acc:>10.3f}{acc-cf:>+8.3f}  {bar}')

rng = np.random.default_rng(9)
M = 6000
true_p = rng.uniform(0.5, 1.0, M)                    # 真实的「这条判对的概率」
correct = (rng.random(M) < true_p).astype(float)
conf_overconf = np.clip(0.5 + (true_p - 0.5) * 1.9, 0, 1)      # 过度自信
conf_good = true_p                                              # 完美校准
conf_useless = np.full(M, 0.8)                                  # 无信息

for name, c in [('完美校准', conf_good), ('过度自信', conf_overconf), ('无信息置信度', conf_useless)]:
    e, _ = expected_calibration_error(c, correct)
    print(f'{name:<16} ECE = {e:.4f}')

e_good, _ = expected_calibration_error(conf_good, correct)
e_over, rows_over = expected_calibration_error(conf_overconf, correct)
assert e_over > e_good + 0.03
print('\n过度自信的可靠性图（▼ 表示实际准确率低于自报置信度）:')
reliability_table(rows_over)

In [ ]:
def temperature_scale(conf, T):
    """对置信度做温度缩放：先转 logit，除以 T，再转回概率。T>1 削弱自信。"""
    c = np.clip(np.asarray(conf, dtype=float), 1e-6, 1 - 1e-6)
    z = np.log(c / (1 - c)) / T
    return 1 / (1 + np.exp(-z))

def fit_temperature(conf, correct, grid=None):
    grid = grid if grid is not None else np.linspace(0.5, 4.0, 71)
    best = min(grid, key=lambda T: expected_calibration_error(temperature_scale(conf, T), correct)[0])
    return float(best)

T_hat = fit_temperature(conf_overconf, correct)
e_before, _ = expected_calibration_error(conf_overconf, correct)
e_after, _ = expected_calibration_error(temperature_scale(conf_overconf, T_hat), correct)
print(f'拟合温度 T = {T_hat:.2f}（>1 表示原本过度自信）')
print(f'ECE: {e_before:.4f} → {e_after:.4f}')
assert T_hat > 1.0 and e_after < e_before
print('\n✅ 温度缩放只改置信度、不改判断——所以它不会影响一致率，只让置信度可用。')

# 更实用的置信度：用 swap 一致性代替自报置信度
# 注意：swap 一致性本身不是概率，要在留出集上校准成概率（这一步几乎免费）
rng = np.random.default_rng(13)
swap_consistent = (rng.random(M) < (0.55 + 0.45 * (true_p - 0.5) * 2)).astype(float)
half = M // 2
p_cons = float(correct[:half][swap_consistent[:half] == 1].mean())      # 留出集上估计
p_incons = float(correct[:half][swap_consistent[:half] == 0].mean())
print(f'留出集校准: swap 一致 → 实际准确率 {p_cons:.3f} | swap 不一致 → {p_incons:.3f}')
conf_from_swap = np.where(swap_consistent == 1, p_cons, p_incons)
e_swap, _ = expected_calibration_error(conf_from_swap, correct)
print(f'用 swap 一致性构造的置信度: ECE = {e_swap:.4f}（对比自报过度自信 {e_before:.4f}）')
assert e_swap < e_before
print('✅ 从行为里测出来的置信度，通常比模型自报的更校准——而且边际成本为零')
print('   （做 swap 探针时已经算出来了）。')

In [ ]:
# 校准的真正用途：分诊。低置信度的 10% 里包含了多少错误？
def triage_gain(conf, correct, frac=0.10):
    idx = np.argsort(conf)                                  # 置信度从低到高
    k = max(int(len(conf) * frac), 1)
    lowest = idx[:k]
    errors_total = float((1 - correct).sum())
    errors_caught = float((1 - correct[lowest]).sum())
    return errors_caught / errors_total if errors_total else 0.0

for name, c in [('完美校准', conf_good), ('过度自信', conf_overconf),
                ('swap 一致性', conf_from_swap), ('无信息', conf_useless)]:
    print(f'{name:<16} 复核最低置信的 10% → 捕捉到 {triage_gain(c, correct):.1%} 的错误')

assert triage_gain(conf_good, correct) > triage_gain(conf_useless, correct) + 0.05
print('\n✅ 校准良好的置信度能让 10% 的人力捕捉到远超 10% 的错误——')
print('   这是把有限人力回报最大化的直接方法（呼应 C66-02 的三级复核流水线）。')

## 5 · 成本-一致性前沿：便宜 judge + 大样本 vs 贵 judge + 小样本

In [ ]:
JUDGES = [
    # name,           相对成本, 单条一致率
    ('小模型 + 整体分',    1.0, 0.68),
    ('小模型 + rubric',    1.3, 0.76),
    ('小模型 + rubric+CoT', 2.2, 0.80),
    ('大模型 + 整体分',    6.0, 0.79),
    ('大模型 + rubric+CoT', 12.0, 0.87),
    ('大模型 + 3-judge 集成', 36.0, 0.89),
]

def system_z(p_agree, delta, n):
    """系统级结论的 z 值 ∝ (2p-1)*delta*sqrt(n)。"""
    return (2 * p_agree - 1) * delta * math.sqrt(n)

BUDGET = 6000.0        # 单位：相对成本
print(f"{'judge 配置':<24}{'成本':>7}{'一致率':>9}{'可跑样本':>10}{'系统级 z':>12}")
rows = []
for name, c, p_ in JUDGES:
    n_ = BUDGET / c
    z_ = system_z(p_, 0.05, n_)
    rows.append((name, c, p_, n_, z_))
    print(f'{name:<24}{c:>7.1f}{p_:>9.0%}{n_:>10,.0f}{z_:>12.2f}')

best = max(rows, key=lambda r: r[4])
print(f'\n同样 {BUDGET:,.0f} 的预算下，系统级分辨力最高的是: **{best[0]}**（z = {best[4]:.2f}）')
assert best[0] != '大模型 + 3-judge 集成', '最贵的配置在固定预算下反而不是最优'
print('✅ 最贵的配置在固定预算下反而落后——因为样本量被成本压掉了。')
print('   ⚠️ 但这只对**系统级**用途成立。如果 judge 的单条判断会被直接消费')
print('      （训练信号、分诊、给用户看），那就必须看一致率本身，不能拿样本量换。')

In [ ]:
# 便宜手段的顺序：rubric 化 → CoT → swap → 换大模型
UPGRADES = [
    ('起点：小模型 + 整体分',   1.0, 0.68),
    ('+ rubric 化',            1.3, 0.76),
    ('+ CoT',                  2.2, 0.80),
    ('+ swap 双跑',            4.4, 0.82),
    ('+ 换大模型',            26.4, 0.88),
]
print(f"{'升级步骤':<24}{'累计成本':>10}{'一致率':>9}{'每倍成本换来的提升':>20}")
prev_c, prev_p = None, None
for name, c, p_ in UPGRADES:
    if prev_c is None:
        print(f'{name:<24}{c:>10.1f}{p_:>9.0%}{"—":>20}')
    else:
        gain_per_cost = (p_ - prev_p) / (c / prev_c)
        print(f'{name:<24}{c:>10.1f}{p_:>9.0%}{gain_per_cost:>19.3f}')
    prev_c, prev_p = c, p_

rubric_eff = (0.76 - 0.68) / (1.3 / 1.0)
bigmodel_eff = (0.88 - 0.82) / (26.4 / 4.4)
print(f'\nrubric 化的单位成本收益 {rubric_eff:.3f} vs 换大模型 {bigmodel_eff:.3f}')
assert rubric_eff > 5 * bigmodel_eff
print('✅ rubric 化的性价比是换大模型的十几倍——')
print('   常见错误是跳过前三步直接换最贵的模型，结果买到的提升还不如免费的 rubric 化。')

## 6 · 漂移检测：CUSUM 抓缓慢漂移

In [ ]:
def cusum(values, baseline, k=0.5, h=5.0, sigma=1.0):
    """单边（向下）CUSUM。k 是允许的漂移容差（以 sigma 为单位），h 是告警阈值。
    返回 (累积和序列, 首次告警的下标或 None)。"""
    s, out, alarm = 0.0, [], None
    for i, v in enumerate(values):
        z = (baseline - v) / sigma                 # 向下漂移为正
        s = max(0.0, s + z - k)
        out.append(s)
        if alarm is None and s > h:
            alarm = i
    return out, alarm

rng = np.random.default_rng(23)
BASE, SIGMA, T = 0.82, 0.02, 40
# 场景一：稳定
stable = rng.normal(BASE, SIGMA, T)
# 场景二：缓慢漂移（从第 15 期起每期掉 0.004）
slow = stable.copy()
slow[15:] -= np.arange(1, T - 15 + 1) * 0.004
# 场景三：突变（第 25 期掉 0.06）
jump = stable.copy()
jump[25:] -= 0.06

THRESH = BASE - 2 * SIGMA
for name, series in [('稳定', stable), ('缓慢漂移', slow), ('突变', jump)]:
    _, alarm = cusum(series, BASE, k=0.5, h=5.0, sigma=SIGMA)
    single = next((i for i, v in enumerate(series) if v < THRESH), None)
    print(f'{name:<10} CUSUM 首次告警于第 {str(alarm):>5} 期 | 单点阈值首次告警于第 {str(single):>5} 期')

_, a_stable = cusum(stable, BASE, sigma=SIGMA)
_, a_slow = cusum(slow, BASE, sigma=SIGMA)
_, a_jump = cusum(jump, BASE, sigma=SIGMA)
single_stable = next((i for i, v in enumerate(stable) if v < THRESH), None)

assert a_stable is None, 'CUSUM 在稳定序列上不应误报'
assert single_stable is not None, '单点阈值在稳定序列上就已经误报了'
assert a_slow is not None and a_jump is not None, 'CUSUM 必须抓到漂移与突变'
print('\n✅ 关键在第一行：**单点阈值在稳定序列上就已经告警了**（纯粹是噪声撞到阈值），')
print('   所以它在另外两条序列上的「早期告警」同样没有信息量——它一直在喊狼来了。')
print('   CUSUM 在稳定序列上保持沉默，在缓慢漂移开始后 7 期、突变后 2 期告警。')
print('\n   哨兵集的纪律：它必须永远不变，而且**绝不能被用来调 judge prompt**——')
print('   一旦被用作优化目标，它就再也不能告诉你任何关于漂移的事。')

## ✏️ 练习 1：从三个数字算出元评测卡的一行

实现 `agreement_card(a_jh, a_hh, a_chance=0.5)`：返回字典
`{'a_jh', 'a_hh', 'ceiling_normalized', 'p_human', 'p_judge', 'verdict'}`，
其中 `verdict` ∈ `{'at_ceiling', 'usable', 'weak'}`：
归一化 ≥ 0.9 → `at_ceiling`；≥ 0.7 → `usable`；否则 `weak`。

In [ ]:
def agreement_card(a_jh, a_hh, a_chance=0.5):
    # TODO：复用 p_from_hh / p_judge_from_jh / ceiling_normalized
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
c1 = agreement_card(0.82, 0.84)
assert c1['verdict'] == 'at_ceiling'
assert abs(c1['ceiling_normalized'] - 0.9412) < 1e-3
c2 = agreement_card(0.70, 0.90)
assert c2['verdict'] == 'weak'
c3 = agreement_card(0.78, 0.88)
assert c3['verdict'] == 'usable'
for name, c in [('judge 0.82 / 人类 0.84', c1), ('judge 0.70 / 人类 0.90', c2),
                ('judge 0.78 / 人类 0.88', c3)]:
    print(f"{name:<24} 归一化 {c['ceiling_normalized']:.3f} | p_J {c['p_judge']:.3f} | {c['verdict']}")
print('✅ 练习 1 通过：同样是「一致率 0.82」，天花板不同，结论完全不同。')

## ✏️ 练习 2：kappa 与原始一致率的分歧幅度

实现 `kappa_gap(base_rate, p_agree, n=20000, seed=0)`：
构造一个正类比例为 `base_rate`、两个评判者各以 `p_agree` 的概率与真值一致的场景，
返回 `(原始一致率, kappa, 两者之差)`。用它画出「不均衡程度 → kappa 塌陷」的曲线。

In [ ]:
def kappa_gap(base_rate, p_agree, n=20000, seed=0):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
acc_bal, kap_bal, gap_bal = kappa_gap(0.50, 0.90, seed=1)
acc_imb, kap_imb, gap_imb = kappa_gap(0.95, 0.90, seed=1)
assert gap_imb > gap_bal, '越不均衡，原始一致率与 kappa 的差距越大'
assert abs(acc_bal - acc_imb) < 0.03, '两种场景的原始一致率其实差不多'
print(f"{'正类比例':>10}{'原始一致率':>12}{'kappa':>10}{'差':>10}")
for br in [0.50, 0.70, 0.85, 0.95, 0.99]:
    a_, k_, g_ = kappa_gap(br, 0.90, seed=1)
    print(f'{br:>10.0%}{a_:>12.3f}{k_:>10.3f}{g_:>10.3f}')
print('✅ 练习 2 通过：原始一致率几乎不随不均衡程度变化，kappa 却一路塌陷——')
print('   两个数字必须一起报，只报其中一个都会误导。')

## ✏️ 练习 3：给定预算的最优 judge 配置

实现 `best_judge_under_budget(judges, budget, delta, use_case)`：
`judges` 是 `[(name, cost, p_agree)]`；`use_case ∈ {'system', 'sample'}`。
- `system`：按 `system_z(p, delta, budget/cost)` 最大化
- `sample`：单条判断被直接消费，按 `p_agree` 最大化（成本只用来过滤买不起的）

返回胜出的 `(name, cost, p_agree)`。

In [ ]:
def best_judge_under_budget(judges, budget, delta, use_case):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
b_sys = best_judge_under_budget(JUDGES, 6000.0, 0.05, 'system')
b_smp = best_judge_under_budget(JUDGES, 6000.0, 0.05, 'sample')
print('系统级用途（选型/排名）  →', b_sys)
print('样本级用途（训练信号/分诊）→', b_smp)
assert b_sys[0] != b_smp[0], '两种用途应当选出不同的 judge'
assert b_smp[2] == max(p for _, _, p in JUDGES), '样本级用途必须选一致率最高的'
assert b_sys[1] < b_smp[1], '系统级用途会选更便宜的（省下的钱换样本量）'
print('✅ 练习 3 通过：「哪个 judge 更好」这个问题，在没说用途时没有答案。')

## ✏️ 练习 4：分诊收益曲线

实现 `triage_curve(conf, correct, fracs)`：对每个 `frac` 返回
`(frac, 捕捉到的错误比例, 提升倍数)`，提升倍数 = 捕捉比例 / frac
（= 1 表示与随机抽检无异）。

In [ ]:
def triage_curve(conf, correct, fracs):
    # TODO：复用上面的 triage_gain
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
FR = [0.05, 0.10, 0.20, 0.50]
cur_good = triage_curve(conf_good, correct, FR)
cur_useless = triage_curve(conf_useless, correct, FR)
print(f"{'复核比例':>10}{'完美校准: 捕捉/倍数':>26}{'无信息: 捕捉/倍数':>24}")
for (f1, c1_, m1), (f2, c2_, m2) in zip(cur_good, cur_useless):
    print(f'{f1:>10.0%}{c1_:>16.1%} / {m1:>5.2f}x{c2_:>16.1%} / {m2:>5.2f}x')
assert cur_good[0][2] > 1.5, '好的置信度在小比例复核时提升倍数应显著大于 1'
assert abs(cur_useless[0][2] - 1.0) < 0.3, '无信息置信度的提升倍数应接近 1'
assert cur_good[0][2] > cur_good[-1][2], '提升倍数随复核比例增大而下降（必然趋近 1）'
print('✅ 练习 4 通过：提升倍数就是「这个置信度值不值得用来分诊」的直接读数。')
print('   接近 1 说明它跟随机抽检没区别——那这个置信度不该被使用。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def agreement_card(a_jh, a_hh, a_chance=0.5):
    ph = p_from_hh(a_hh)
    pj = p_judge_from_jh(a_jh, ph)
    cn = ceiling_normalized(a_jh, a_hh, a_chance)
    verdict = 'at_ceiling' if cn >= 0.9 else ('usable' if cn >= 0.7 else 'weak')
    return {'a_jh': a_jh, 'a_hh': a_hh, 'ceiling_normalized': cn,
            'p_human': ph, 'p_judge': pj, 'verdict': verdict}

In [ ]:
# 练习 2 参考答案
def kappa_gap(base_rate, p_agree, n=20000, seed=0):
    rng = np.random.default_rng(seed)
    truth = (rng.random(n) < base_rate).astype(int)
    r1 = np.where(rng.random(n) < p_agree, truth, 1 - truth)
    r2 = np.where(rng.random(n) < p_agree, truth, 1 - truth)
    acc = float((r1 == r2).mean())
    kap = cohen_kappa(r1, r2)
    return (acc, kap, acc - kap)

In [ ]:
# 练习 3 参考答案
def best_judge_under_budget(judges, budget, delta, use_case):
    affordable = [(nm, c, p) for nm, c, p in judges if c <= budget]
    if not affordable:
        return None
    if use_case == 'system':
        return max(affordable, key=lambda t: system_z(t[2], delta, budget / t[1]))
    if use_case == 'sample':
        return max(affordable, key=lambda t: t[2])
    raise ValueError(use_case)

In [ ]:
# 练习 4 参考答案
def triage_curve(conf, correct, fracs):
    out = []
    for f in fracs:
        caught = triage_gain(conf, correct, f)
        out.append((f, caught, caught / f if f > 0 else float('nan')))
    return out

---
## 🧪 真实工程胶囊：元评测的落地清单

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════
# A. 元评测集的三段结构（分开报告，绝不混算）
# ══════════════════════════════════════════════════════════════════
# main_set/       随机抽样 400-600 条，3 位标注员 → 多数意见   → 回答「整体有多准」
# hard_set/       分歧驱动 150-250 条（swap 不一致 / 两个 judge 意见不同）→ 「难的地方有多准」
# probe_set/      对抗构造（长度/格式/署名对照，模块 02）      → 「有没有系统偏差」
# sentinel_set/   固定 200 条，永不改动，只用于漂移监测        → 「judge 变了没有」
#
# 纪律：sentinel_set 的结果只看不改。一旦拿它去调 prompt，它就失去监测价值。

# ══════════════════════════════════════════════════════════════════
# B. 标注收集的最小规范
# ══════════════════════════════════════════════════════════════════
# 1. 每条至少 3 个标注员（要算 a_HH，2 个是下限，3 个才能取多数）
# 2. 标注界面**随机化 A/B 顺序**（否则人类标注也带位置偏差）
# 3. 标注员看不到模型名（否则带品牌偏差）
# 4. 记录标注耗时——耗时长的样本就是难样本，天然是 hard_set 的候选
# 5. 先做一轮 20 条的校准会，对齐口径，再正式标注

# ══════════════════════════════════════════════════════════════════
# C. 每次评测都跑的哨兵检查（接进 CI，见 C68-04）
# ══════════════════════════════════════════════════════════════════
def sentinel_check(judge, sentinel, baseline):
    res = run_judge(judge, sentinel)
    agreement = (res.verdicts == sentinel.human_labels).mean()
    se = (agreement * (1 - agreement) / len(sentinel)) ** 0.5
    checks = {
        "agreement_drop":  baseline.agreement - agreement > 2 * se,
        "mean_shift":      abs(res.scores.mean() - baseline.mean) > 3 * baseline.se,
        "swap_drop":       baseline.swap_consistency - res.swap_consistency > 0.03,
        "parse_fail_up":   res.parse_fail_rate > baseline.parse_fail_rate * 2,
    }
    return checks       # 任一为 True → 阻断本次评测，先查 judge

# ══════════════════════════════════════════════════════════════════
# D. 元评测卡模板（贴进 eval card）
# ══════════════════════════════════════════════════════════════════
# judge–human / human–human / chance / ceiling-normalized / kappa
# 方向一致率（tie 视为可接受）
# 系统级：与人类排名的 Spearman
# 校准：ECE + 低置信 10% 捕捉到的错误比例
# 漂移：相对基线的 Δ 与 CUSUM 状态
'''
print(RECIPE)

### 小结

| 你学到的 | 一句话 | 用在哪 |
|---|---|---|
| 对齐目标 | 用更强的模型当金标准只能证明一致，不能证明对 | 元评测设计 |
| 人类上界 | 一致率必须配着 a_HH 和随机一致率一起读 | 每份报告 |
| kappa 悖论 | 不均衡分布上 kappa 塌陷，是信号不是缺陷 | 选系数 |
| 样本级 vs 系统级 | 一致率 70% 也能给出正确排名——**前提是误差随机** | 判断 judge 能不能用 |
| 校准与分诊 | swap 一致性比自报置信度更校准，且边际成本为零 | 人力分配 |
| 成本-一致性 | rubric 化的性价比是换大模型的十几倍 | 选 judge |
| CUSUM 哨兵 | 抓缓慢漂移；哨兵集永远不能被用来调 prompt | 长期运维 |

下一模块：**04 · 从成对比较到排名**——Bradley–Terry、Elo、置信区间、主动配对，
以及「排行榜上相差 20 分到底算不算差距」。